# ISLES'24 Fold 4 → Kaggle handoff — sharded + resumable

This CPU-only notebook avoids one giant 3+ GiB tar. It builds **small persistent tar shards** (10 subjects per shard), stores each finished shard in Drive, and skips completed shards on rerun.

A persistent log is written to `MyDrive/HybridStrokeSeg/ISLES2024/results/kaggle_handoff_sharded.log`, so if anything fails I can inspect the exact error afterward.


In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RESULTS = ROOT/'results'
SHARDS = ROOT/'kaggle_handoff_shards'
SCRATCH = Path('/content/isles2024_kaggle_scratch')
LOG = RESULTS/'kaggle_handoff_sharded.log'
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')
RESULTS.mkdir(parents=True,exist_ok=True)
SHARDS.mkdir(parents=True,exist_ok=True)
SCRATCH.mkdir(parents=True,exist_ok=True)

def run_logged(cmd, cwd=None):
    cmd=[str(x) for x in cmd]
    print('$',' '.join(cmd),flush=True)
    with LOG.open('a',encoding='utf-8') as h:
        h.write('\n$ '+' '.join(cmd)+'\n'); h.flush()
        p=subprocess.Popen(cmd,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        assert p.stdout is not None
        for line in p.stdout:
            print(line,end=''); h.write(line); h.flush()
        rc=p.wait()
        if rc!=0: raise subprocess.CalledProcessError(rc,cmd)

LOG.write_text('ISLES24 Kaggle sharded handoff started\n',encoding='utf-8')


## Load the latest repository


In [ ]:
if REPO.exists(): shutil.rmtree(REPO)
if REPO_ZIP.exists(): REPO_ZIP.unlink()
run_logged([
    'curl','-fL','--retry','20','--retry-delay','3','--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o',REPO_ZIP
])
run_logged(['unzip','-q',REPO_ZIP,'-d','/content'])
os.chdir(REPO)
run_logged([sys.executable,'-m','pip','install','-q','-e','.'])


## Build/resume the persistent shards

Each finished shard is copied back to Drive with a SHA256 sidecar. If the runtime disconnects, rerun this notebook and completed shards will be reused.


In [ ]:
run_logged([
    sys.executable,'scripts/make_isles2024_kaggle_shards.py',
    '--preprocessed-root',ROOT/'nnUNet_preprocessed',
    '--results-root',ROOT/'nnUNet_results',
    '--output-dir',SHARDS,
    '--scratch-dir',SCRATCH,
    '--fold','4',
    '--subjects-per-shard','10'
],cwd=REPO)

print('Shard files:')
for p in sorted(SHARDS.iterdir()):
    if p.is_file(): print(p.name, round(p.stat().st_size/(1024**2),1),'MiB')


## Authenticate to Kaggle

Upload your small `kaggle.json` token when prompted.


In [ ]:
uploaded=files.upload()
if 'kaggle.json' not in uploaded: raise RuntimeError('Please upload kaggle.json')
KD=Path.home()/'.kaggle'; KD.mkdir(exist_ok=True)
(KD/'kaggle.json').write_bytes(uploaded['kaggle.json'])
os.chmod(KD/'kaggle.json',0o600)
run_logged([sys.executable,'-m','pip','install','-q','kaggle'])
run_logged(['kaggle','config','view'])


## Upload the shard folder as a private Kaggle Dataset

The upload folder is flat: small tar shards, SHA256 files, manifest, and metadata. No directory packaging is needed.


In [ ]:
creds=json.loads((KD/'kaggle.json').read_text())
username=creds['username']
metadata={
  'title':'ISLES2024 NCCT Fold4 Resume Shards - Private',
  'id':f'{username}/isles2024-ncct-fold4-resume-shards',
  'licenses':[{'name':'other'}]
}
(SHARDS/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2))

check=subprocess.run(['kaggle','datasets','files',metadata['id']],capture_output=True,text=True)
if check.returncode==0:
    run_logged(['kaggle','datasets','version','-p',SHARDS,'-m','Updated sharded Fold 4 resume','-r','skip','--ignore-patterns','*.partial'])
else:
    run_logged(['kaggle','datasets','create','-p',SHARDS,'-r','skip','--ignore-patterns','*.partial'])
print('Private Kaggle Dataset:',metadata['id'])


## Next

On Kaggle, create a GPU notebook, attach the private Dataset `isles2024-ncct-fold4-resume-shards`, and run repository notebook `16_kaggle_resume_isles2024_fold4_SHARDED.ipynb`.
